In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

train = pd.read_csv("data/clean/train.csv")
test = pd.read_csv("data/clean/test_scored.csv")
cutoff = 548

inf = np.inf
bins = {
    "utilisation":       [-inf, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0, inf],
    "age":               [-inf, 30, 40, 50, 60, 70, inf],
    "late_30_59":        [-inf, 0, 1, 2, inf],
    "late_60_89":        [-inf, 0, 1, inf],
    "late_90":           [-inf, 0, 1, inf],
    "debt_ratio":        [-inf, 0.2, 0.4, 0.6, 0.8, 1.0, inf],
    "monthly_income":    [-inf, 2000, 3500, 5000, 7500, 10000, inf],
    "open_credit_lines": [-inf, 2, 5, 10, 15, inf],
    "real_estate_loans": [-inf, 0, 1, 2, inf],
    "dependents":        [-inf, 0, 1, 2, inf],
}

def make_bins(data, col):
    b = pd.cut(data[col], bins[col])
    return b.cat.add_categories("Missing").fillna("Missing")

def woe_table(data, col):
    b = make_bins(data, col)
    t = data.groupby(b, observed=False)["default_2yrs"].agg(total="size", bad="sum")
    t["good"] = t["total"] - t["bad"]
    t = t[t["total"] > 0]
    pct_good = (t["good"] + 0.5) / t["good"].sum()
    pct_bad = (t["bad"] + 0.5) / t["bad"].sum()
    t["woe"] = np.log(pct_good / pct_bad)
    t.index = t.index.astype(str)
    return t

def to_woe(data, cols):
    out = pd.DataFrame(index=data.index)
    for col in cols:
        out[col] = make_bins(data, col).astype(str).map(woe_table(train, col)["woe"])
    return out

In [2]:
test["age_group"] = pd.cut(test["age"], [0, 29, 44, 59, 120], labels=["18–29", "30–44", "45–59", "60+"])
test["approved_with_age"] = test["score"] >= cutoff

g = test.groupby("age_group", observed=True).agg(
    applicants=("age", "size"),
    actual_bad_rate=("default_2yrs", "mean"),
    approval_rate=("approved_with_age", "mean"))
g["impact_ratio"] = g["approval_rate"] / g["approval_rate"].max()
g[["actual_bad_rate", "approval_rate"]] *= 100
g.round(2)

,applicants,actual_bad_rate,approval_rate,impact_ratio
age_group,,,,
18–29,2654,12.40,74.76,0.78
30–44,11755,9.70,79.52,0.83
45–59,16044,6.97,86.74,0.91
60+,14547,2.89,95.72,1.00


In [3]:
feats_no_age = [f for f in bins if f != "age"]
m2 = LogisticRegression(max_iter=1000).fit(to_woe(train, feats_no_age), train["default_2yrs"])
p2 = m2.predict_proba(to_woe(test, feats_no_age))[:, 1]

gini_no_age = 2 * roc_auc_score(test["default_2yrs"], p2) - 1
approval_target = test["approved_with_age"].mean()
test["approved_no_age"] = p2 <= np.quantile(p2, approval_target)

print("Gini with age:    0.716")
print("Gini without age:", round(gini_no_age, 3))
print("Approval rate (both):", round(approval_target * 100, 1), "% vs", round(test["approved_no_age"].mean() * 100, 1), "%")

Gini with age:    0.716
Gini without age: 0.709
Approval rate (both): 87.1 % vs 87.1 %


In [4]:
def summarise(d):
    return pd.Series({
        "approval_with_age": d["approved_with_age"].mean() * 100,
        "approval_no_age": d["approved_no_age"].mean() * 100,
        "bad_rate_approved_with_age": d.loc[d["approved_with_age"], "default_2yrs"].mean() * 100,
        "bad_rate_approved_no_age": d.loc[d["approved_no_age"], "default_2yrs"].mean() * 100})

comp = test.groupby("age_group", observed=True)[["approved_with_age", "approved_no_age", "default_2yrs"]].apply(summarise).round(1)
comp

,approval_with_age,approval_no_age,bad_rate_approved_with_age,bad_rate_approved_no_age
age_group,,,,
18–29,74.8,77.7,5.3,5.5
30–44,79.5,80.8,4.0,4.1
45–59,86.7,86.5,3.3,3.3
60+,95.7,94.4,1.7,1.6


In [5]:
def profit_per_10k(mask):
    bads = test.loc[mask, "default_2yrs"].sum()
    goods = mask.sum() - bads
    return (goods * 400 - bads * 3000) / len(test) * 10000

p_with = profit_per_10k(test["approved_with_age"])
p_without = profit_per_10k(test["approved_no_age"])
print(f"Profit with age:    £{p_with:,.0f}")
print(f"Profit without age: £{p_without:,.0f}")
print(f"Cost of removing age: £{p_with - p_without:,.0f} per 10k applicants ({(p_with - p_without) / p_with * 100:.1f}%)")

comp.to_csv("outputs/fairness_by_age.csv")

Profit with age:    £2,595,022
Profit without age: £2,588,978
Cost of removing age: £6,044 per 10k applicants (0.2%)
